# Model-based gate: does a PFN-only lookahead find the headroom over EI?

## Description

<!-- Written 2026-11-11 before the first run (pre-registration). Design: research_log/decisions/0003 §8.3–8.4. Issue #13. -->

**Question.** The api_round1 pilot (notebook 06) found sizeable *clairvoyant* headroom over EI's pick: the cross-fitted best of 16 labeled candidates gains 0.70 [0.57, 0.84] % of the gap per state, against EI's final regret of 1.34. The learned head could not exploit it.

Here is the cheaper and sharper question first: does a lookahead that uses **only the PFN** recognise which candidate beats EI's pick? The lookahead is a one-step improvement whose continuations are simulated from the PFN's own predictive. Such a policy is deployable, though expensive. Its gain bounds what amortizing it (the value head / decoder) can reach without ground truth. If it finds nothing, the headroom isn't visible from the belief, and amortized policy iteration on this prior has little to gain from the PFN state.

**Setup.**
- *Belief continuation* (`experimental/api_gate.py`): from state (D, t), query the candidate, draw its y from the PFN's posterior predictive, follow EI to T with every y drawn from the PFN given the fantasized data, and record the best y.
  - 4 continuations per candidate.
  - Nuisance (candidate sets, fantasy uniforms) is shared across candidates, so the comparison uses common random numbers.
  - The replicate ids are disjoint from those of the ground-truth labels, so the model's choice shares no luck with the label that scores it.
- *Stage 1a (states).* On the first 2000 labeled states of `api_round1/pilot_20261111-133311` (split 4, d = 2), score the same 16 candidates per state.
  - Select the best model advantage (x₀ if none > 0), without the task.
  - Score the selected candidate with its *ground-truth* label (the mean over 4 true-task continuations).
  - Mean over states: G = a deployable, first-order per-state gain.
  - References: the clairvoyant cross-fitted best (0.70 on all 4000 states; recomputed here on the same 2000), and a random valid candidate (null: deviation without information).
- *Stage 1b (policy).* The model-based rollout policy against EI on 200 split-1 tasks, paired (common random numbers), d = 2, n_init = 4, T = 20.
  - At every step: the round-1 candidate set, the same 16-candidate choice rule, 4 belief continuations each.
  - Query the best candidate if its model advantage is > 0, otherwise EI's pick.

**What to look for.**
1. Stage 1b: the paired Δ = mean_task [R(mb) − R(EI)] in % of the gap, with bootstrap 95% CI, and the deviation rate.
2. Stage 1a:
   - G and its CI, as a share of the clairvoyant reference;
   - G at stricter thresholds (model advantage > τ·se, τ ∈ {1, 2}), secondary;
   - the within-state Spearman between model and true advantage;
   - G by branch step t.

**GO / NO-GO** (rule `analysis/api.py::gate_verdict`, on Stage 1b, worthwhile = 5% relative):
- **GO**: the model-based policy lowers mean normalized regret by ≥ 5% with the CI of Δ below 0.
- **NO-GO**: the CI rules out a 5% gain.
- **INCONCLUSIVE**: otherwise.

GO means the PFN's belief carries the planning information, so amortizing it is worthwhile. Then the readout probes (notebook 08) ask *where* in the PFN state it sits, and the 0006 decoder tries to learn it. NO-GO with Stage 1a G ≈ 0 means the headroom of notebook 06 is invisible to the PFN's own belief. It is then PFN error or luck, not plannable from the belief, and fine-tuning the PFN on BO contexts (0004/0005, E1) becomes the more promising lever.

**Limits.**
- Noise-free P0, d = 2, T = 20 only. `pfn_large` is valid for n ≤ 100; contexts here have n ≤ 24.
- Belief continuations sample marginals step by step, re-encoding after each sample: the PFN's posterior predictive process, not a joint sample.
- 16 candidates per decision and 4 continuations: a cheap ceiling. A larger candidate set or more continuations could find more.
- 200 test tasks is a pilot size.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/api_gate/<sweep_id>"  # set after `just snapshot runs/api_gate/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.